# MDP Lesson 1: discounted MDP  

This lesson can be downloaded as a notebook, a notebook for colab and a python file [here](https://marmote.gitlabpages.inria.fr/marmote/python_downloads.html)

This C++ notebook is the Xeus-cling counterpart of the Python lesson. It follows the same pedagogical progression while using the official Marmote C++ API.

## Using the library

**Import the modules**

The following commands allow one to configure Xeus-cling and to include the headers required for a discounted MDP.

In [1]:
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMDP")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMarkovChain.dll")
#pragma cling load("marmoteMDP.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMDP")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")
#pragma cling load("libmarmoteMDP.so")
#endif

#include <iostream>
#include <string>
#include <vector>

#include <marmoteCore/marmoteFullMatrix.h>
#include <marmoteCore/marmoteInterval.h>
#include <marmoteCore/marmoteSparseMatrix.h>
#include <marmoteMDP/marmoteDiscountedMDP.h>
#include <marmoteMDP/marmoteFeedbackSolutionMDP.h>

using namespace std;

It is necessary to import the modules corresponding to *marmote.core* and *marmote.mdp* from the C++ Marmote library.  
Hence the core headers handle the basic objects (*Sets*, *distributions*, *matrices*, ...), while the MDP headers handle Markov Decision Process objects.

In this first lesson, we show how to make and how to solve a simple infinite-horizon discounted criteria MDP.

## Build a simple MDP

### Reminders about  MDP

Formally a MDP is a tuple *(S,A,P_a,R)* where  
+ S is the state space  
+ A is the action space  
+ P_a is a collection of transition matrices. A matrix for each action  
+ R is a reward (or cost) matrix  

#### Description of the example implemented here

We assume a simple model with two states x1=0 and x2=1 and in each state: two actions a1=0 and a2=1.

The reward matrix is:

|      |      |
|:----:|:----:|
| 4.5  | 2.0  |
| -1.5 | 3.0  |

where *r(x,a)* is the entry with row coordinate $x$ and column coordinate $a$.  
The entry *r(x,a)* represents the reward when in state *x* action *a* is performed.

The transition matrices are:

Transition matrix of the action 0:

|      |      |
|:----:|:----:|
| 0.6  | 0.4  |
| 0.5  | 0.5  |

Transition matrix of the action 1:

|      |      |
|:----:|:----:|
| 0.2  | 0.8  |
| 0.7  | 0.3  |

### Elements of a MDP object

#### Attributes of the object

A MDP in Marmote receives at least four important attributes:

1. The *state space* that is a *MarmoteSet* object.
2. The *action space* that is also a *MarmoteSet* object.
3. A vector of transition structures.  
   Each entry in the vector corresponds with a transition structure associated with a given action.
4. A reward structure, preferably a `FullMatrix`, whose entry `(x,a)` is the reward of action `a` in state `x`.

#### How to build the DiscountedMDP object

**Create state space and action space**

Here we define two `MarmoteInterval` objects for state and action spaces. The two following lines define two intervals going from 0 to 1.

In [2]:
MarmoteSet* actionSpace = new MarmoteInterval(0, 1);
MarmoteSet* stateSpace = new MarmoteInterval(0, 1);

**Storing matrices**

A `vector` is used in C++ to store all transition matrices.  
The number of matrices should correspond with the size of the action space.

In [3]:
vector<TransitionStructure*> trans(actionSpace->Cardinal());

**Build transition matrices**

Now, we create `P0`, an object `SparseMatrix` with size 2x2. The command to initialize an entry is `setEntry(row, column, value)`.

In [4]:
SparseMatrix* P0 = new SparseMatrix(2);
P0->setEntry(0, 0, 0.6);
P0->setEntry(0, 1, 0.4);
P0->setEntry(1, 0, 0.5);
P0->setEntry(1, 1, 0.5);
cout << "Matrix P0" << endl;
P0->Write(&cout);
trans.at(0) = P0;

Matrix P0
         0          0 6.000000e-01
         0          1 4.000000e-01
         1          0 5.000000e-01
         1          1 5.000000e-01


We now add a new `SparseMatrix` for the second action.

In [5]:
SparseMatrix* P1 = new SparseMatrix(2);
P1->setEntry(0, 0, 0.2);
P1->setEntry(0, 1, 0.8);
P1->setEntry(1, 0, 0.7);
P1->setEntry(1, 1, 0.3);
trans.at(1) = P1;

**Build reward matrix**

We create a `FullMatrix` object with size 2x2. This matrix is used for storing the rewards associated with each couple *(state, action)*.

In [6]:
FullMatrix* Reward = new FullMatrix(2, 2);
Reward->setEntry(0, 0, 4.5);
Reward->setEntry(0, 1, 2.0);
Reward->setEntry(1, 0, -1.5);
Reward->setEntry(1, 1, 3.0);

**Additional parameters of the discounted MDP**

Two additional parameters should be entered: *beta* and *criterion*.  
- *beta* is the discount factor for incorporating future values. In the C++ code below, we store it in a variable named `discountFactor` because `beta` is ambiguous in Xeus-cling due to `std::beta`.  
- *criterion* indicates the optimisation criterion, which is either maximisation (`"max"`) or minimisation (`"min"`).

In [7]:
double discountFactor = 0.95;
string criterion = "max";

**Build a discounted MDP**

We now construct a `DiscountedMDP` object using the parameters defined above.

In [8]:
DiscountedMDP* mdp = new DiscountedMDP(criterion, stateSpace, actionSpace, trans, Reward, discountFactor);
mdp->Write();

#############################################
Model: Infinite Horizon MDP
MDP Criteria : Infinite horizon discounted
Discount factor:0.95
#############################################
#############################################
Model: Infinite Horizon MDP
MDP Criteria : Infinite horizon discounted
Discount factor:0.95
DiscountedMDP (Object at 0x56123398bb10)
MDP type (discrete,continuous): discrete
MDP rule (min,max): max
State space size: 2
Action space size: 2
State  dimension: 1
Action dimension: 1
#############################################
Transition matrix per action:
action: 0
         0          0 6.000000e-01
         0          1 4.000000e-01
         1          0 5.000000e-01
         1          1 5.000000e-01

action: 1
         0          0 2.000000e-01
         0          1 8.000000e-01
         1          0 7.000000e-01
         1          1 3.000000e-01

#############################################
Reward Matrix (state,action):
         0          0 4.500000e+00
  

## Solving the MDP

### List of solution methods

We list here the different methods implemented to solve discounted Markov Decision Processes. All these methods return a `FeedbackSolutionMDP` object.

1. *Value Iteration* with method name `ValueIteration`  
2. *Value Iteration using Gauss Seidel* with method name `ValueIterationGS`   
3. *Value Iteration with a given initial value function* with method name `ValueIterationInit`   
4. *Policy Iteration Modified* with method name `PolicyIterationModified`  
5. *Policy Iteration Modified with Gauss Seidel* with method name `PolicyIterationModifiedGS`

### Running solution methods

Parameters:  
1. *epsilon* is a precision threshold used to determine convergence.  
2. *maxIter* gives the maximal number of authorised iterations.

In [9]:
double epsilon = 0.00001;
int maxIter = 700;

**Value Iteration**

In [10]:
FeedbackSolutionMDP* optimum = mdp->ValueIteration(epsilon, maxIter);
optimum->Write();

#############################################
Solution of MDP problem
Size of the state space: 2
#############################################
Solution model: Feedback Stationary Policy
- column 1: index of the state
- column 2: Value function 
- column 3: Optimal action 

  0            79.589   0
  1           78.2192   1
#############################################



**Gauss Seidel Value Iteration**

In [11]:
FeedbackSolutionMDP* optimum2 = mdp->ValueIterationGS(epsilon, 10);
optimum2->Write();

#############################################
Solution of MDP problem
Size of the state space: 2
#############################################
Solution model: Feedback Stationary Policy
- column 1: index of the state
- column 2: Value function 
- column 3: Optimal action 

  0           38.9873   0
  1           39.3571   1
#############################################



**Value Iteration Init**

It is also possible to choose which value function will be used to start the value iteration process. To do this, one enters a third parameter which is a `SolutionMDP*` whose value field is used to initialise the process.

In [12]:
FeedbackSolutionMDP* optimum3 = mdp->ValueIterationInit(epsilon, 200, optimum2);
optimum3->Write();

#############################################
Solution of MDP problem
Size of the state space: 2
#############################################
Solution model: Feedback Stationary Policy
- column 1: index of the state
- column 2: Value function 
- column 3: Optimal action 

  0           79.5876   0
  1           78.2178   1
#############################################



**Policy Iteration Modified**

In [13]:
FeedbackSolutionMDP* optimum4 = mdp->PolicyIterationModified(epsilon, maxIter, 0.001, 100);
optimum4->Write();

#############################################
Solution of MDP problem
Size of the state space: 2
#############################################
Solution model: Feedback Stationary Policy
- column 1: index of the state
- column 2: Value function 
- column 3: Optimal action 

  0            79.589   0
  1           78.2192   1
#############################################



**Policy Iteration Modified with Gauss Seidel**

In [14]:
FeedbackSolutionMDP* optimum5 = mdp->PolicyIterationModifiedGS(epsilon, maxIter, 0.001, 100);
optimum5->Write();

#############################################
Solution of MDP problem
Size of the state space: 2
#############################################
Solution model: Feedback Stationary Policy
- column 1: index of the state
- column 2: Value function 
- column 3: Optimal action 

  0            79.589   0
  1           78.2192   1
#############################################



## About the SolutionMDP object

The solution is stored in a `FeedbackSolutionMDP` object. This object has attributes that store a value and an action for each state.  
The printing of a `FeedbackSolutionMDP` gives first information about the policy, and then displays the information for all states.

The next instructions illustrate how to read the values and the actions after solving the MDP.

In [15]:
mdp->PolicyCost(optimum, epsilon, maxIter);
double* values = optimum->getValue();
stateType* actions = optimum->getAction();
for (int i = 0; i < stateSpace->Cardinal(); ++i) {
    cout << "i= " << i << " value= " << values[i] << " action= " << actions[i] << endl;
}

i= 0 value= 7.958904e+01 action= 0
i= 1 value= 7.821917e+01 action= 1


End of the notebook

In [16]:
delete optimum;
delete optimum2;
delete optimum3;
delete optimum4;
delete optimum5;
delete mdp;
delete stateSpace;
delete actionSpace;